<a href="https://colab.research.google.com/github/llazdll/Spark_BookRating/blob/main/BookRating.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [28]:

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, format_number, concat, substring, to_timestamp, date_format,count



In [29]:

spark=SparkSession.builder.appName("Book Rating").master("local[*]").getOrCreate()


In [30]:
!wget -O Book-Ratings.csv https://raw.githubusercontent.com/llazdll/Spark_BookRating/main/BooksRating-CSV/Book-Ratings.csv
!wget -O Books.csv https://raw.githubusercontent.com/llazdll/Spark_BookRating/main/BooksRating-CSV/Books.csv
!wget -O Users.csv https://raw.githubusercontent.com/llazdll/Spark_BookRating/main/BooksRating-CSV/Users.csv

--2026-08-18 12:08:37--  https://raw.githubusercontent.com/llazdll/Spark_BookRating/main/BooksRating-CSV/Book-Ratings.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 30682266 (29M) [application/octet-stream]
Saving to: ‘Book-Ratings.csv’

Book-Ratings.csv    100%[===================>]  29.26M  92.8MB/s    in 0.3s    

2026-08-18 12:08:38 (92.8 MB/s) - ‘Book-Ratings.csv’ saved [30682266/30682266]

--2026-08-18 12:08:38--  https://raw.githubusercontent.com/llazdll/Spark_BookRating/main/BooksRating-CSV/Books.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, await

In [31]:

df_BookRating= spark.read \
    .option("header", "true") \
    .option("delimiter", ";") \
    .option("inferSchema", "true") \
    .csv("Book-Ratings.csv")


In [32]:
df_BookRating.show()
df_BookRating.printSchema()

+------+----------+----+
|userid|      isbn|rate|
+------+----------+----+
|276725|034545104X|   0|
|276726|0155061224|   5|
|276727|0446520802|   0|
|276729|052165615X|   3|
|276729|0521795028|   6|
|276733|2080674722|   0|
|276736|3257224281|   8|
|276737|0600570967|   6|
|276744|038550120X|   7|
|276745| 342310538|  10|
|276746|0425115801|   0|
|276746|0449006522|   0|
|276746|0553561618|   0|
|276746|055356451X|   0|
|276746|0786013990|   0|
|276746|0786014512|   0|
|276747|0060517794|   9|
|276747|0451192001|   0|
|276747|0609801279|   0|
|276747|0671537458|   9|
+------+----------+----+
only showing top 20 rows
root
 |-- userid: integer (nullable = true)
 |-- isbn: string (nullable = true)
 |-- rate: integer (nullable = true)



In [33]:
df_Books= spark.read \
    .option("header", "true") \
    .option("delimiter", ";") \
    .option("inferSchema", "true") \
    .csv("Books.csv")

In [34]:
df_Books.show()
df_Books.printSchema()

+----------+--------------------+--------------------+-----------------+--------------------+--------------------+--------------------+--------------------+
|      ISBN|           BookTitle|          BookAuthor|YearOfPublication|           Publisher|           ImageURLS|           ImageURLM|           ImageURLL|
+----------+--------------------+--------------------+-----------------+--------------------+--------------------+--------------------+--------------------+
|0195153448| Classical Mythology|  Mark P. O. Morford|             2002|Oxford University...|http://images.ama...|http://images.ama...|http://images.ama...|
|0002005018|        Clara Callan|Richard Bruce Wright|             2001|HarperFlamingo Ca...|http://images.ama...|http://images.ama...|http://images.ama...|
|0060973129|Decision in Normandy|        Carlo D'Este|             1991|     HarperPerennial|http://images.ama...|http://images.ama...|http://images.ama...|
|0374157065|Flu: The Story of...|    Gina Bari Kolata|    

In [35]:
df_Users= spark.read \
    .option("header", "true") \
    .option("delimiter", ";") \
    .option("inferSchema", "true") \
    .csv("Users.csv")

In [36]:
df_Users.show()
df_Users.printSchema()

+------+-----------+--------------------+----+
|UserID|   USERNAME|            Location| Age|
+------+-----------+--------------------+----+
|     1|bzsufoRTLN2|  nyc, new york, usa|NULL|
|     2|fq7kfHg4VEI|stockton, califor...|  18|
|     3|W0Hbkd3xR8v|moscow, yukon ter...|NULL|
|     4|W51GahAx5Ap|porto, v.n.gaia, ...|  17|
|     5|VKN3PQ18GgN|farnborough, hant...|NULL|
|     6|h9BSgQZ5wOk|santa monica, cal...|  61|
|     7|7rdddpZpjWp| washington, dc, usa|NULL|
|     8|qiOJebWJS2i|timmins, ontario,...|NULL|
|     9|gkcxQJLS13A|germantown, tenne...|NULL|
|    10|BANPptNSbPy|albacete, wiscons...|  26|
|    11|X0ELjsCzJt0|melbourne, victor...|  14|
|    12|z4qLRN05PYT|fort bragg, calif...|NULL|
|    13|4YgtGJggfAx|barcelona, barcel...|  26|
|    14|FZW1MG3zLpo|mediapolis, iowa,...|NULL|
|    15|FHo7m8eHkPb|calgary, alberta,...|NULL|
|    16|HwLFQdzk30o|albuquerque, new ...|NULL|
|    17|dk15dEDpgbc|chesapeake, virgi...|NULL|
|    18|aNFnRPqjGL1|rio de janeiro, r...|  25|
|    19|hNw8C

In [39]:
df_BookRating.groupBy('isbn').count().show()

+----------+-----+
|      isbn|count|
+----------+-----+
|2080674722|    3|
|3499134004|    1|
|3548603203|   24|
|880781112X|    3|
|0738205737|    3|
|0749317256|    1|
|0515131520|   16|
|8471662531|    1|
|0441005470|    1|
|0739417096|    7|
|0767906373|    8|
|0425087859|    3|
|0553574566|   45|
|0498024253|    2|
|0395977894|   44|
|0505522004|    5|
|0425188787|   60|
|0515137111|   58|
|0767905385|  143|
|0553580906|   41|
+----------+-----+
only showing top 20 rows


datasets

In [43]:
df_BookAvg = df_BookRating.groupBy("isbn").avg("rate")
df_BookAvg.show()

+----------+------------------+
|      isbn|         avg(rate)|
+----------+------------------+
|2080674722|3.6666666666666665|
|3499134004|               0.0|
|3548603203|3.4166666666666665|
|880781112X| 4.333333333333333|
|0738205737|1.6666666666666667|
|0749317256|               0.0|
|0515131520|            1.6875|
|8471662531|               7.0|
|0441005470|               0.0|
|0739417096| 5.285714285714286|
|0767906373|             1.125|
|0425087859|2.6666666666666665|
|0553574566| 2.977777777777778|
|0498024253|               3.5|
|0395977894| 5.090909090909091|
|0505522004|               1.4|
|0425188787|3.4166666666666665|
|0515137111|3.5344827586206895|
|0767905385| 3.825174825174825|
|0553580906| 2.341463414634146|
+----------+------------------+
only showing top 20 rows


In [51]:
df_RatingUser = df_BookRating.join(
    df_Users,
    df_BookRating.userid == df_Users.UserID,
    "inner"
)
df_RatingUser.show()

+------+----------+----+------+-----------+--------------------+----+
|userid|      isbn|rate|UserID|   USERNAME|            Location| Age|
+------+----------+----+------+-----------+--------------------+----+
|     9|0440234743|   0|     9|gkcxQJLS13A|germantown, tenne...|NULL|
|     9|0452264464|   6|     9|gkcxQJLS13A|germantown, tenne...|NULL|
|     9|0609804618|   0|     9|gkcxQJLS13A|germantown, tenne...|NULL|
|    12|1879384493|  10|    12|z4qLRN05PYT|fort bragg, calif...|NULL|
|    16|0345402871|   9|    16|HwLFQdzk30o|albuquerque, new ...|NULL|
|    16|0345417623|   0|    16|HwLFQdzk30o|albuquerque, new ...|NULL|
|    17|0312978383|   0|    17|dk15dEDpgbc|chesapeake, virgi...|NULL|
|    17|0425099148|   7|    17|dk15dEDpgbc|chesapeake, virgi...|NULL|
|    17|0553264990|   5|    17|dk15dEDpgbc|chesapeake, virgi...|NULL|
|    17|0553278398|   0|    17|dk15dEDpgbc|chesapeake, virgi...|NULL|
|    17|0684823802|   0|    17|dk15dEDpgbc|chesapeake, virgi...|NULL|
|    17|0891075275| 

In [53]:
df_RatingUser = df_RatingUser.select(
    df_BookRating.userid,
    df_BookRating.isbn,
    df_BookRating.rate,
    df_Users.USERNAME
)
df_RatingUser.show()

+------+----------+----+-----------+
|userid|      isbn|rate|   USERNAME|
+------+----------+----+-----------+
|276725|034545104X|   0|qP0KgArgnkK|
|276726|0155061224|   5|KD6BOeOQiPM|
|276727|0446520802|   0|AWJdCVnDLd8|
|276729|052165615X|   3|8RRnyqqWTcr|
|276729|0521795028|   6|8RRnyqqWTcr|
|276733|2080674722|   0|imIbzo4rK7T|
|276736|3257224281|   8|4udWNF5VnOd|
|276737|0600570967|   6|LPbrg3STxAe|
|276744|038550120X|   7|CoM54wt6AiU|
|276745| 342310538|  10|jLRV9LYfEUZ|
|276746|0425115801|   0|7uxi3eD6COS|
|276746|0449006522|   0|7uxi3eD6COS|
|276746|0553561618|   0|7uxi3eD6COS|
|276746|055356451X|   0|7uxi3eD6COS|
|276746|0786013990|   0|7uxi3eD6COS|
|276746|0786014512|   0|7uxi3eD6COS|
|276747|0060517794|   9|Eg2O6NDkj3d|
|276747|0451192001|   0|Eg2O6NDkj3d|
|276747|0609801279|   0|Eg2O6NDkj3d|
|276747|0671537458|   9|Eg2O6NDkj3d|
+------+----------+----+-----------+
only showing top 20 rows


In [55]:
df_UserBook = df_RatingUser.join(
    df_Books,
    df_RatingUser.isbn == df_Books.ISBN,
    "inner"
)
df_UserBook.show()

+------+----------+----+-----------+----------+--------------------+--------------------+-----------------+--------------------+--------------------+--------------------+--------------------+
|userid|      isbn|rate|   USERNAME|      ISBN|           BookTitle|          BookAuthor|YearOfPublication|           Publisher|           ImageURLS|           ImageURLM|           ImageURLL|
+------+----------+----+-----------+----------+--------------------+--------------------+-----------------+--------------------+--------------------+--------------------+--------------------+
|171118|0000913154|   8|6chdqlR3DC7|0000913154|The Way Things Wo...|C. van Amerongen ...|             1967|Simon &amp; Schuster|http://images.ama...|http://images.ama...|http://images.ama...|
| 86123|0001010565|   0|px70uymJ7k6|0001010565|     Mog's Christmas|         Judith Kerr|             1992|             Collins|http://images.ama...|http://images.ama...|http://images.ama...|
|209516|0001010565|   0|mjteD2ip2Lj|0001

In [56]:
df_UserBook = df_UserBook.select(
    df_RatingUser.USERNAME,
    df_RatingUser.isbn,
    df_RatingUser.rate,
    df_Books.BookTitle
)

In [58]:
df_Final = df_UserBook.join(
    df_BookAvg,
    df_UserBook.isbn == df_BookAvg.isbn,
    "inner"
)
df_Final.show()

+-----------+----------+----+--------------------+----------+-------+
|   USERNAME|      isbn|rate|           BookTitle|      isbn|BookAvg|
+-----------+----------+----+--------------------+----------+-------+
|6chdqlR3DC7|0000913154|   8|The Way Things Wo...|0000913154|    8.0|
|px70uymJ7k6|0001010565|   0|     Mog's Christmas|0001010565|    0.0|
|mjteD2ip2Lj|0001010565|   0|     Mog's Christmas|0001010565|    0.0|
|cHwJip4Kj4k|0001046438|   9|                Liar|0001046438|    9.0|
|6VUiynjA3tV|0001046934|   0|The Prime of Miss...|0001046934|    0.0|
|cHwJip4Kj4k|0001047213|   9|    The Fighting Man|0001047213|    9.0|
|a0EEWhgtsW8|0001047647|   0|  First Among Equals|0001047647|    0.0|
|Tupz6KKVgIq|0001047663|   0|    Matter Of Honour|0001047663|    0.0|
|OMqCFWvTBPp|0001047868|   0|           Kidnapped|0001047868|    0.0|
|EM5BvtuvZ91|0001047973|   9|     Brave New World|0001047973|    9.0|
|cHwJip4Kj4k|0001047973|   9|     Brave New World|0001047973|    9.0|
|cHwJip4Kj4k|0001048

sql

In [44]:
df_BookRating.createOrReplaceTempView("BookRating")
df_Books.createOrReplaceTempView("Books")
df_Users.createOrReplaceTempView("Users")

In [49]:
df_BookAvg = spark.sql("""
    SELECT isbn, AVG(rate) AS BookAvg
    FROM BookRating
    GROUP BY isbn
""")

df_BookAvg.show()

+----------+------------------+
|      isbn|           BookAvg|
+----------+------------------+
|2080674722|3.6666666666666665|
|3499134004|               0.0|
|3548603203|3.4166666666666665|
|880781112X| 4.333333333333333|
|0738205737|1.6666666666666667|
|0749317256|               0.0|
|0515131520|            1.6875|
|8471662531|               7.0|
|0441005470|               0.0|
|0739417096| 5.285714285714286|
|0767906373|             1.125|
|0425087859|2.6666666666666665|
|0553574566| 2.977777777777778|
|0498024253|               3.5|
|0395977894| 5.090909090909091|
|0505522004|               1.4|
|0425188787|3.4166666666666665|
|0515137111|3.5344827586206895|
|0767905385| 3.825174825174825|
|0553580906| 2.341463414634146|
+----------+------------------+
only showing top 20 rows


In [47]:
df_full_details = spark.sql("""
    SELECT
        u.USERNAME,
        b.BookTitle,
        r.rate,
        AVG(r.rate) OVER (PARTITION BY r.isbn) AS BookAvg
    FROM BookRating r
    JOIN Users u
        ON r.userid = u.UserID
    JOIN Books b
        ON r.isbn = b.ISBN
""")
df_full_details.show()

+-----------+--------------------+----+-------+
|   USERNAME|           BookTitle|rate|BookAvg|
+-----------+--------------------+----+-------+
|6chdqlR3DC7|The Way Things Wo...|   8|    8.0|
|px70uymJ7k6|     Mog's Christmas|   0|    0.0|
|mjteD2ip2Lj|     Mog's Christmas|   0|    0.0|
|cHwJip4Kj4k|                Liar|   9|    9.0|
|6VUiynjA3tV|The Prime of Miss...|   0|    0.0|
|cHwJip4Kj4k|    The Fighting Man|   9|    9.0|
|a0EEWhgtsW8|  First Among Equals|   0|    0.0|
|Tupz6KKVgIq|    Matter Of Honour|   0|    0.0|
|OMqCFWvTBPp|           Kidnapped|   0|    0.0|
|EM5BvtuvZ91|     Brave New World|   9|    9.0|
|cHwJip4Kj4k|     Brave New World|   9|    9.0|
|cHwJip4Kj4k|Nothing Can Be Be...|   0|    0.0|
|SyGcdQu7P4o|        Dark Spectre|   0|    0.0|
|e8Hb8GyY8yN| Pearl and Sir Orfeo|   5|    5.0|
|krizZUAK9f9|Cereus Blooms At ...|   8|    8.0|
|LCv9LN0AQaf|CHESS FOR YOUNG B...|   8|    8.0|
|3X2htHp8Jgk|Paddington's Birt...|   0|    0.0|
|3X2htHp8Jgk|Paddington in the...|   0| 